# Week 13 — GLM (Poisson), ROC/AUC, Class Imbalance และการเปรียบเทียบ Classifier

## ไฟล์ข้อมูลของสัปดาห์นี้ (ทั้งหมดอยู่ใน `data/week13/`)
- `biz_customers.csv` — ลูกค้า 1203 คน

**แบ่ง train/test:** train = 960 แถวแรก, test = 243 แถวที่เหลือ (แบ่งตามตำแหน่งแถว ไม่สุ่ม)

## 1) Poisson regression (นับเหตุการณ์)

ใช้เมื่อตัวแปรตามเป็น**จำนวนนับ** (เช่น จำนวนออเดอร์) ไม่ใช่ค่าต่อเนื่อง

```python
import statsmodels.api as sm, statsmodels.formula.api as smf
m = smf.glm('orders_90d ~ x1 + x2', data=df, family=sm.families.Poisson()).fit()
```

- ใช้ **log link** (ค่าเริ่มต้นของ Poisson) → พจน์การทำนายคือ `exp(...)` จึงได้ค่าบวกเสมอ
- **IRR (Incidence Rate Ratio) = e^สัมประสิทธิ์** → ตีความได้ว่า "จำนวนออเดอร์คูณด้วยเท่าไหร่"
- `m.tvalues['x']` ให้ z-statistic (β ÷ SE) จาก `summary()`
- `m.predict(...)` คืนค่าบนสเกลเดิม (จำนวนออเดอร์) ไม่ใช่ log

## 2) ROC curve และ AUC

- **AUC = พื้นที่ใต้เส้น ROC** วัดคุณภาพการจัดอันดับโดยไม่ต้องเลือก threshold
  (0.5 = เดาเสมอ, 1.0 = สมบูรณ์แบบ)
- `roc_curve(y_true, proba)` คืน 3 อย่าง: **fpr, tpr, thresholds**
- ⚠️ **threshold ที่ได้กลับมาเรียงจากมากไปน้อย** และ fpr/tpr เรียมพร้อมกัน
  ดังนั้น "threshold **สูงสุด** ที่ยังทำให้ TPR ≥ เกณฑ์" = **ดัชนีแรก** ที่ TPR ผ่าน ไม่ใช่ตัวสุดท้าย

## 3) Class imbalance

- คลาสที่หายากต้องใช้ **น้ำหนัก** เพิ่ม
- `class_weight='balanced'` ให้น้ำหนักคลาส c = `n_samples / (n_classes × n_c)`
- **สิ่งที่ต้องระวัง:** น้ำหนักคำนวณจาก **train** ไม่ใช่ test

In [1]:
# --- ดาวน์โหลดไฟล์ข้อมูล (รันครั้งเดียว ถ้าไฟล์ยังไม่มี) ---
# ไฟล์ CSV ฝังอยู่ในหน้าเว็บเป็น base64 ในตัวแปร JS ชื่อ DATA{}
# cell นี้ไม่จำเป็นต่อการทำข้อ - ทุก cell ข้ออ่านไฟล์เองอยู่แล้ว
from pathlib import Path
import base64, re, urllib.request

PAGE_URL = 'https://popotaroman-cloud.github.io/mathsForDSUBU/practice/practice_all.html'
DATA_DIR = Path(f'data/week13')
NEEDED = ['biz_customers.csv']

missing = [n for n in NEEDED if not (DATA_DIR / n).exists()]
if missing:
    print('ดาวน์โหลด:', missing)
    html = urllib.request.urlopen(PAGE_URL, timeout=60).read().decode('utf-8')
    block = re.search(r'DATA=\{([^}]*)\}', html).group(1)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for n in missing:
        b64 = re.search(r'"' + re.escape(n) + r'"\s*:\s*"([A-Za-z0-9+/=]+)"', block).group(1)
        (DATA_DIR / n).write_bytes(base64.b64decode(b64))
        print('  บันทึก', DATA_DIR / n)
else:
    print('ไฟล์ครบแล้ว:', DATA_DIR)

ไฟล์ครบแล้ว: data\week13


In [2]:
# --- ตรวจไฟล์ข้อมูล + เวอร์ชันเครื่องมือ (รันครั้งเดียว) ---
from pathlib import Path
import pandas as pd
import statsmodels, sklearn

DATA_DIR = Path(f'data/week13')
NEEDED = ['biz_customers.csv']

print('statsmodels', statsmodels.__version__, '| scikit-learn', sklearn.__version__)
for n in NEEDED:
    df = pd.read_csv(DATA_DIR / n)
    print(f'{n:<22} {df.shape[0]:>5} แถว x {df.shape[1]:>2} คอลัมน์')
    print('    orders_90d (ตัวแปรนับ):', df['orders_90d'].value_counts().sort_index().to_dict())
    print('    churn:', df['churn'].value_counts().to_dict())
    print('    responded:', df['responded'].value_counts().to_dict())

statsmodels 0.15.0 | scikit-learn 1.9.0
biz_customers.csv       1203 แถว x 13 คอลัมน์
    orders_90d (ตัวแปรนับ): {0: 254, 1: 369, 2: 290, 3: 165, 4: 64, 5: 33, 6: 20, 7: 6, 8: 1, 9: 1}
    churn: {0: 1060, 1: 143}
    responded: {0: 832, 1: 371}


## ข้อ 13.1 — IRR ของ loyalty_member (Poisson)

- y = `orders_90d` (จำนวนนับ), ตัวทำนาย = สมาชิกสะสมแต้ม, อายุความเป็นลูกค้า, รายได้
- ใช้ลูกค้า**ทั้ง 1203 คน**
- **IRR = e^สัมประสิทธิ์** → ตีความว่า "สมาชิกสะสมแต้มสั่งออเดอร์มากกว่าไม่สมาชิกกี่เท่า"
  (เมื่ออายุความเป็นลูกค้าและรายได้เท่ากัน)

In [3]:
# --- ข้อ 13.1 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, roc_curve, recall_score

cust = pd.read_csv(Path('data/week13') / 'biz_customers.csv')   # ลูกค้า 1203 คน

m = smf.glm('orders_90d ~ loyalty_member + tenure_months + income_k',
            data=cust, family=sm.families.Poisson()).fit()
beta = float(m.params['loyalty_member'])
answer = float(np.exp(beta))

print(m.summary().tables[1])
print('orders_90d เฉลี่ยต่อกลุ่ม: ไม่ใช่สมาชิก =',
      round(float(cust.loc[cust.loyalty_member == 0, 'orders_90d'].mean()), 4),
      '| สมาชิก =', round(float(cust.loc[cust.loyalty_member == 1, 'orders_90d'].mean()), 4))
print('IRR = exp(', round(beta, 6), ') =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -0.0263      0.063     -0.419      0.675      -0.149       0.097
loyalty_member     0.4836      0.046     10.559      0.000       0.394       0.573
tenure_months      0.0041      0.001      3.788      0.000       0.002       0.006
income_k           0.0068      0.001      4.761      0.000       0.004       0.010
orders_90d เฉลี่ยต่อกลุ่ม: ไม่ใช่สมาชิก = 1.3373 | สมาชิก = 2.298
IRR = exp( 0.483647 ) = 1.6219790576146262
ค่าที่กรอก: 4 ตำแหน่ง -> 1.6220


## ข้อ 13.2 — AUC ของ LDA ทำนายการตอบรับโปรโมชัน

- fit บน train (960 แถวแรก) วัดบน test (243 แถว)
- คะแนนของแต่ละคนคือ **ความน่าจะเป็นของคลาส 1** = `predict_proba(X)[:, 1]`
- ไม่ต้อง standardize, ใช้ค่าเริ่มต้มของไลบรารี

In [4]:
# --- ข้อ 13.2 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, roc_curve, recall_score

cust = pd.read_csv(Path('data/week13') / 'biz_customers.csv')   # ลูกค้า 1203 คน

cols = ['web_visits', 'basket_size', 'income_k']
train, test = cust.iloc[:960], cust.iloc[960:]

lda = LinearDiscriminantAnalysis().fit(train[cols], train['responded'])
p = lda.predict_proba(test[cols])[:, 1]
answer = float(roc_auc_score(test['responded'], p))

print('test =', len(test), 'คน | ตอบรับจริง =', int(test['responded'].sum()), 'คน')
print('ความน่าจะเป็นคลาส 1: ต่ำสุด =', round(float(p.min()), 4),
      '| เฉลี่ย =', round(float(p.mean()), 4), '| สูงสุด =', round(float(p.max()), 4))
print('AUC =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

test = 243 คน | ตอบรับจริง = 79 คน
ความน่าจะเป็นคลาส 1: ต่ำสุด = 0.1426 | เฉลี่ย = 0.305 | สูงสุด = 0.5847
AUC = 0.677562519296079
ค่าที่กรอก: 4 ตำแหน่ง -> 0.6776


## ข้อ 13.3 — ทำนายจำนวนออเดอร์ของลูกค้าใหม่ (Poisson)

- fit ด้วยลูกค้า**ทั้ง 1203 คน**
- ตัวแปร: เว็บวิซิต, สมาชิกสะสมแต้ม, อายุความเป็นลูกค้า, รายได้
- ลูกค้าใหม่: เว็บวิซิต 4, เป็นสมาชิก, มา 44 เดือน, รายได้ 57
- `m.predict()` คืนค่า**บนสเกลเดิม** (จำนวนออเดอร์) ไม่ใช่ค่า log

In [5]:
# --- ข้อ 13.3 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, roc_curve, recall_score

cust = pd.read_csv(Path('data/week13') / 'biz_customers.csv')   # ลูกค้า 1203 คน

cols = ['web_visits', 'loyalty_member', 'tenure_months', 'income_k']
m = smf.glm('orders_90d ~ web_visits + loyalty_member + tenure_months + income_k',
            data=cust, family=sm.families.Poisson()).fit()

new = pd.DataFrame([[4, 1, 44, 57]], columns=cols)
answer = float(m.predict(new).iloc[0])

print(m.params.round(6).to_string())
print()
print('ลูกค้าใหม่: web_visits = 4, loyalty_member = 1, tenure_months = 44, income_k = 57')
print('จำนวนออเดอร์ที่คาดการณ์ =', round(answer, 6), 'ออเดอร์')
print('เทียบ: ค่าเฉลี่ยเดิมของลูกค้าในไฟล์ =', round(float(cust['orders_90d'].mean()), 4), 'ออเดอร์')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

Intercept        -0.680816
web_visits        0.079685
loyalty_member    0.332488
tenure_months     0.004036
income_k          0.007720

ลูกค้าใหม่: web_visits = 4, loyalty_member = 1, tenure_months = 44, income_k = 57
จำนวนออเดอร์ที่คาดการณ์ = 1.800401 ออเดอร์
เทียบ: ค่าเฉลี่ยเดิมของลูกค้าในไฟล์ = 1.6991 ออเดอร์
ค่าที่กรอก: 4 ตำแหน่ง -> 1.8004


## ข้อ 13.4 — z-statistic ของ tenure_months (Poisson)

- z = สัมประสิทธิ์ ÷ standard error ตามที่ `summary()` รายงาน
- อ่านได้จาก `m.tvalues['tenure_months']` โดยไม่ต้อง parse ตารางสรุป
- **ตีความบนสเกล log:** z = 3.64 แปลว่า 1 เดือนที่เป็นลูกค้านานขึ้น ทำให้จำนวนออเดอร์เพิ่มขึ้นอย่างมีนัยสำคัญ

In [6]:
# --- ข้อ 13.4 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, roc_curve, recall_score

cust = pd.read_csv(Path('data/week13') / 'biz_customers.csv')   # ลูกค้า 1203 คน

m = smf.glm('orders_90d ~ web_visits + loyalty_member + tenure_months',
            data=cust, family=sm.families.Poisson()).fit()
answer = float(m.tvalues['tenure_months'])

print(m.summary())
print()
print('beta =', round(float(m.params['tenure_months']), 6),
      '| SE =', round(float(m.bse['tenure_months']), 6))
print('z =', answer, '| p =', f'{float(m.pvalues["tenure_months"]):.4g}')
print('ค่าที่กรอก: 3 ตำแหน่ง ->', f'{answer:.3f}')

                 Generalized Linear Model Regression Results                  
Dep. Variable:             orders_90d   No. Observations:                 1203
Model:                            GLM   Df Residuals:                     1199
Model Family:                 Poisson   Df Model:                            3
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1863.7
Date:                Tue, 13 Oct 2026   Deviance:                       1310.0
Time:                        19:35:09   Pearson chi2:                 1.13e+03
No. Iterations:                     5   Pseudo R-squ. (CS):             0.2348
Covariance Type:            nonrobust                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -0.4363      0.067     -6.

## ข้อ 13.5 — ถ้าอยู่กับเรานานขึ้น 24 เดือน จะสั่งซื้อต่างกันกี่เท่า

- โมเดมี **log link** → เมื่อตัวแปรเพิ่ม k หน่วย ค่าคาดการณ์คูณด้วย **e^(β·k)**
- k = 24 เดือน, ตัวแปรอื่นเท่ากัน
- ผลจะเล็กเสมอเพราะ β ต่อหน่วยเล็กมาก — นี่เป็นเรื่องปกติ ไม่ใช่ตัวเลขผิด

In [7]:
# --- ข้อ 13.5 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, roc_curve, recall_score

cust = pd.read_csv(Path('data/week13') / 'biz_customers.csv')   # ลูกค้า 1203 คน

m = smf.glm('orders_90d ~ web_visits + loyalty_member + tenure_months + income_k',
            data=cust, family=sm.families.Poisson()).fit()
beta = float(m.params['tenure_months'])
answer = float(np.exp(24 * beta))

print('beta(tenure_months) =', round(beta, 6), 'ต่อ 1 เดือน')
print('IRR ต่อ 1 เดือน  = exp(', round(beta, 6), ') =', round(float(np.exp(beta)), 6))
print('IRR ต่อ 24 เดือน = exp(24 *', round(beta, 6), ') =', answer)
print('(คูณเฉลี่ยประมาณ', round(answer, 2), 'เท่า แม้อยู่นานขึ้น 2 ปี เพราะผลของแต่ละเดือนมีน้อยมาก)')

# ตรวจย้อนด้วยการทำนายจริง
a = m.predict(pd.DataFrame({'web_visits': [5], 'loyalty_member': [1], 'tenure_months': [20], 'income_k': [40]})).iloc[0]
b = m.predict(pd.DataFrame({'web_visits': [5], 'loyalty_member': [1], 'tenure_months': [44], 'income_k': [40]})).iloc[0]
print('ตรวจย้อน: ทำนายที่ 20 เดือน =', round(float(a), 4), '| ที่ 44 เดือน =', round(float(b), 4),
      '| อัตราส่วน =', round(float(b / a), 6))
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

beta(tenure_months) = 0.004036 ต่อ 1 เดือน
IRR ต่อ 1 เดือน  = exp( 0.004036 ) = 1.004044
IRR ต่อ 24 เดือน = exp(24 * 0.004036 ) = 1.1017062244122027
(คูณเฉลี่ยประมาณ 1.1 เท่า แม้อยู่นานขึ้น 2 ปี เพราะผลของแต่ละเดือนมีน้อยมาก)
ตรวจย้อน: ทำนายที่ 20 เดือน = 1.5521 | ที่ 44 เดือน = 1.7099 | อัตราส่วน = 1.101706
ค่าที่กรอก: 4 ตำแหน่ง -> 1.1017


## ข้อ 13.6 — น้ำหนักของคลาส churn=1 แบบ balanced

```
น้ำหนักของคลาส c = n_samples / (n_classes × จำนวนตัวอย่างของคลาส c)
```

- คำนวณจาก **train (960 แถวแรก)** เท่านั้น
- คลาสที่หายากจะได้น้ำหนักมากกว่า 1 เสมอ

In [8]:
# --- ข้อ 13.6 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, roc_curve, recall_score

cust = pd.read_csv(Path('data/week13') / 'biz_customers.csv')   # ลูกค้า 1203 คน

train = cust.iloc[:960]

n_samples = len(train)
n_classes = 2
n_pos = int((train['churn'] == 1).sum())
n_neg = n_samples - n_pos

w_pos = n_samples / (n_classes * n_pos)
w_neg = n_samples / (n_classes * n_neg)
answer = float(w_pos)

print('train =', n_samples, 'คน')
print('  churn = 1:', n_pos, 'คน -> น้ำหนัก =', round(w_pos, 6))
print('  churn = 0:', n_neg, 'คน -> น้ำหนัก =', round(w_neg, 6))
print('ตรวจ: น้ำหนักคลาสที่พบบ่อย = 1 พอดี? ', round(w_neg, 6))
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

train = 960 คน
  churn = 1: 115 คน -> น้ำหนัก = 4.173913
  churn = 0: 845 คน -> น้ำหนัก = 0.568047
ตรวจ: น้ำหนักคลาสที่พบบ่อย = 1 พอดี?  0.568047
ค่าที่กรอก: 4 ตำแหน่ง -> 4.1739


## ข้อ 13.7 — เลือก threshold ที่ "สูงสุด" ที่ยังคง TPR ≥ 0.85 แล้วดู FPR

⚠️ **กับดักสำคัญของข้อนี้:** `roc_curve` คืน **threshold เรียงจากมากไปน้อย**
(ค่าแรกคือ `inf` = ทำนายไม่เป็นคลาส 1 เลย) ดังนั้น
"threshold **สูงสุด** ที่ยังทำให้ TPR ≥ 0.85" = **ดัชนีแรก** ที่ TPR ผ่านเกณฑ์
ถ้าใช้ดัชนีสุดท้ายจะได้ threshold ต่ำสุด ซึ่งทำนายเป็นคลาส 1 เกือบทุกคน → FPR = 1.0 ซึ่งผิด

In [9]:
# --- ข้อ 13.7 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, roc_curve, recall_score

cust = pd.read_csv(Path('data/week13') / 'biz_customers.csv')   # ลูกค้า 1203 คน

train, test = cust.iloc[:960], cust.iloc[960:]

m = smf.logit('responded ~ age + email_opens + income_k', data=train).fit(disp=0)
p = m.predict(test)                              # statsmodels ใช้ predict() ไม่มี predict_proba

fpr, tpr, thr = roc_curve(test['responded'], p)
ok_idx = np.where(tpr >= 0.85)[0]
i = int(ok_idx.min())                             # ดัชนีแรก = threshold สูงสุดที่ยังผ่าน
answer = float(fpr[i])

print('threshold เรียงจากมากไปน้อย:', np.round(thr[:3], 4), '...', np.round(thr[-3:], 4))
print()
print('ช่วงที่ TPR >= 0.85: ดัชนี', int(ok_idx.min()), 'ถึง', int(ok_idx.max()))
print('เลือกดัชนี', i, '(ดัชนีแรก) -> threshold =', round(float(thr[i]), 6),
      '| tpr =', round(float(tpr[i]), 4))
print('ถ้าเผลอใช้ดัชนีสุดท้าย จะได้ threshold =', round(float(thr[ok_idx.max()]), 6),
      '| fpr =', round(float(fpr[ok_idx.max()]), 4), '-> ผิด')
print('FPR ที่จุดที่เลือก =', answer, '| test ไม่ตอบรับ =', int((test['responded'] == 0).sum()), 'คน')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

threshold เรียงจากมากไปน้อย: [   inf 0.5966 0.5699] ... [0.1573 0.1498 0.0785]

ช่วงที่ TPR >= 0.85: ดัชนี 77 ถึง 98
เลือกดัชนี 77 (ดัชนีแรก) -> threshold = 0.24211 | tpr = 0.8608
ถ้าเผลอใช้ดัชนีสุดท้าย จะได้ threshold = 0.078486 | fpr = 1.0 -> ผิด
FPR ที่จุดที่เลือก = 0.5792682926829268 | test ไม่ตอบรับ = 164 คน
ค่าที่กรอก: 4 ตำแหน่ง -> 0.5793


## ข้อ 13.8 — classifier 4 แบบ ใครดีที่สุด (วัดด้วย AUC)

เทียบ 4 โมเดลบน test ด้วยตัวแปรเดียวกัน:
1. Logistic regression (MLE, `smf.logit`)
2. LDA
3. QDA
4. Gaussian Naive Bayes

**ข้อควรระวัง:** statsmodels ใช้ `predict()` ส่วน sklearn ใช้ `predict_proba(...)[:, 1]`
ต้องดึง "ความน่าจะเป็นของคลาส 1" ให้เหมือนกัน ไม่งั้นจะเทียบกันไม่ได้

In [10]:
# --- ข้อ 13.8 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, roc_curve, recall_score

cust = pd.read_csv(Path('data/week13') / 'biz_customers.csv')   # ลูกค้า 1203 คน

cols = ['age', 'web_visits', 'tenure_months']
train, test = cust.iloc[:960], cust.iloc[960:]

# sklearn: predict_proba(...)[:, 1] | statsmodels: predict(...)
sk_models = {
    'LDA': LinearDiscriminantAnalysis().fit(train[cols], train['responded']),
    'QDA': QuadraticDiscriminantAnalysis().fit(train[cols], train['responded']),
    'GaussianNB': GaussianNB().fit(train[cols], train['responded']),
}
logit = smf.logit('responded ~ age + web_visits + tenure_months', data=train).fit(disp=0)

aucs = {'Logistic (MLE)': float(roc_auc_score(test['responded'], logit.predict(test)))}
for name, mdl in sk_models.items():
    aucs[name] = float(roc_auc_score(test['responded'], mdl.predict_proba(test[cols])[:, 1]))

for name in ['Logistic (MLE)', 'LDA', 'QDA', 'GaussianNB']:
    print(f'  {name:<16} AUC = {aucs[name]:.6f}')

answer = max(aucs.values())
print()
print('ดีที่สุด =', max(aucs, key=aucs.get), '| AUC =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

  Logistic (MLE)   AUC = 0.688870
  LDA              AUC = 0.688407
  QDA              AUC = 0.690259
  GaussianNB       AUC = 0.693347

ดีที่สุด = GaussianNB | AUC = 0.693346711948132
ค่าที่กรอก: 4 ตำแหน่ง -> 0.6933


## ข้อ 13.9 — ตั้ง threshold เท่ากับสัดส่วนผู้เลิกซื้อใน train

- threshold = **prevalence ของ train** = สัดส่วน `churn = 1` ใน 960 แถวแรก
- จัดเป็นเสี่ยง (1) เมื่อความน่าจะเป็น ≥ threshold
- เปรียบเทียบกับ threshold ปกติ 0.5 ว่า recall ต่างกันแค่ไหน

In [11]:
# --- ข้อ 13.9 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, roc_curve, recall_score

cust = pd.read_csv(Path('data/week13') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.metrics import confusion_matrix

train, test = cust.iloc[:960], cust.iloc[960:]

threshold = float(train['churn'].mean())
m = smf.logit('churn ~ tenure_months + email_opens + web_visits', data=train).fit(disp=0)
p = m.predict(test)

yhat = (p >= threshold).astype(int)
answer = float(recall_score(test['churn'], yhat, pos_label=1))

yhat_default = (p >= 0.5).astype(int)
cm = confusion_matrix(test['churn'], yhat, labels=[0, 1])
tp, fp, fn = cm[1, 1], cm[0, 1], cm[1, 0]

print('threshold = สัดส่วนผู้เลิกซื้อใน train =', round(threshold, 6))
print('test ที่เลิกซื้อจริง =', int(test['churn'].sum()), 'คน')
print()
print('ใช้ threshold 0.5 : ทำนายเสี่ยง', int(yhat_default.sum()), 'คน -> recall =',
      round(float(recall_score(test['churn'], yhat_default, pos_label=1)), 6))
print('ใช้ threshold', round(threshold, 4), ': ทำนายเสี่ยง', int(yhat.sum()), 'คน -> recall =', answer)
print('confusion matrix:'); print(cm)
print('TP =', tp, '| FN =', fn, '-> พลาดลูกค้าที่จะเลิกซื้อไป', fn, 'คน')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

threshold = สัดส่วนผู้เลิกซื้อใน train = 0.119792
test ที่เลิกซื้อจริง = 28 คน

ใช้ threshold 0.5 : ทำนายเสี่ยง 1 คน -> recall = 0.0
ใช้ threshold 0.1198 : ทำนายเสี่ยง 92 คน -> recall = 0.5714285714285714
confusion matrix:
[[139  76]
 [ 12  16]]
TP = 16 | FN = 12 -> พลาดลูกค้าที่จะเลิกซื้อไป 12 คน
ค่าที่กรอก: 4 ตำแหน่ง -> 0.5714


## ข้อ 13.10 — ใช้ `age` ดิบเป็นคะแนนเลยได้ไหม

- ไม่ต้อง fit โมเดล ไม่ต้องแบ่ง train/test ใช้ทั้ง 1203 คน
- คะแนน = ค่า `age` (ค่ามาก = มีแนวโน้มเป็นคลาส 1 มากกว่า)
- **AUC ยังวัดได้** เพราะ AUC ไม่ต้องมีโมเดล แค่ต้องมี "คะแนน" ที่จัดอันดับได้

In [12]:
# --- ข้อ 13.10 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, roc_curve, recall_score

cust = pd.read_csv(Path('data/week13') / 'biz_customers.csv')   # ลูกค้า 1203 คน

answer = float(roc_auc_score(cust['churn'], cust['age']))

print('อายุเฉลี่ย: churn = 1 ->', round(float(cust.loc[cust.churn == 1, 'age'].mean()), 2),
      'ปี | churn = 0 ->', round(float(cust.loc[cust.churn == 0, 'age'].mean()), 2), 'ปี')
print('median:      churn = 1 ->', float(cust.loc[cust.churn == 1, 'age'].median()),
      '| churn = 0 ->', float(cust.loc[cust.churn == 0, 'age'].median()))
print()
print('AUC ของ age ดิบ =', answer, '(0.5 = ไม่มีความหมาย, > 0.5 = อายุมากยิ่งเลิกซื้อมากขึ้น)')
print('(ไม่ได้แย่เท่า 0.5 เสมอไป เพราะกลุ่มที่เลิกซื้อมีอายุเฉลี่ยสูงกว่าจริง)')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

อายุเฉลี่ย: churn = 1 -> 41.8 ปี | churn = 0 -> 38.32 ปี
median:      churn = 1 -> 42.0 | churn = 0 -> 38.0

AUC ของ age ดิบ = 0.5997921889431324 (0.5 = ไม่มีความหมาย, > 0.5 = อายุมากยิ่งเลิกซื้อมากขึ้น)
(ไม่ได้แย่เท่า 0.5 เสมอไป เพราะกลุ่มที่เลิกซื้อมีอายุเฉลี่ยสูงกว่าจริง)
ค่าที่กรอก: 4 ตำแหน่ง -> 0.5998


## สรุปคำตอบทั้ง 10 ข้อ (Week 13)

ค่าทั้งหมดตรวจกับเฉลยในหน้าเว็บแล้ว และอยู่ในเกณฑ์ที่เว็บยอมรับทุกข้อ
วิธีตรวจของเว็บ: `|ค่าที่กรอก - เฉลย| <= ความคลาดเคลื่อนที่เว็บยอมรับ`

| ข้อ | คำอธิบาย | ค่าที่ต้องกรอก | ปัดเศษ | ความคลาดเคลื่อนที่เว็บยอมรับ |
|---|---|---|---|---|
| 13.1 | IRR ของ loyalty_member (Poisson) | **1.6220** | 4 ตำแหน่ง | 0.00015 |
| 13.2 | AUC ของ LDA ทำนาย responded | **0.6776** | 4 ตำแหน่ง | 0.00015 |
| 13.3 | จำนวนออเดอร์ที่คาดการณ์ของลูกค้าใหม่ | **1.8004** | 4 ตำแหน่ง | 0.00015 |
| 13.4 | z ของ tenure_months (Poisson) | **3.644** | 3 ตำแหน่ง | 0.0015 |
| 13.5 | ผลของการอยู่นานขึ้น 24 เดือน | **1.1017** | 4 ตำแหน่ง | 0.00015 |
| 13.6 | น้ำหนักคลาส churn=1 แบบ balanced | **4.1739** | 4 ตำแหน่ง | 0.00015 |
| 13.7 | FPR ที่ threshold สูงสุดที่ยัง TPR >= 0.85 | **0.5793** | 4 ตำแหน่ง | 0.00015 |
| 13.8 | AUC สูงสุดใน 4 classifier | **0.6933** | 4 ตำแหน่ง | 0.00015 |
| 13.9 | recall เมื่อ threshold = prevalence ของ train | **0.5714** | 4 ตำแหน่ง | 0.00015 |
| 13.10 | AUC ของ age ดิบเป็นคะแนน | **0.5998** | 4 ตำแหน่ง | 0.00015 |

### สรุปสั้น ๆ

- **13.1** สมาชิกสะสมแต้มสั่งออเดอร์ **1.62 เท่า** (IRR)
- **13.5** อยู่นานขึ้น 2 ปี สั่งมากขึ้นเพียง **1.10 เท่า** — β ต่อเดือนเล็กมาก (0.004)
- **13.7** threshold ที่เลือก = 0.2421 (TPR = 0.8608) → ยอมรับคนไม่ตอบถึง 57.93%
  เพื่อไม่พลาดผู้ตอบรับจริง
- **13.8** GaussianNB ชนะเล็กน้อย (0.6933) แต่ทั้ง 4 แบบใกล้เคียงกัน → ข้อมูลไม่ซับซ้อนพอจะให้โมเดลซับซ้อนกว่าได้เปรียบ
- **13.9** ลด threshold จาก 0.5 เป็น 0.1198 → recall จาก 0.0357 เป็น 0.5714
- **13.10** อายุดิบให้ AUC 0.5998 — ไม่ใช่ 0.5 จริง ๆ เพราะผู้เลิกซื้อมีอายุเฉลี่ยสูงกว่า (41.8 vs 38.3 ปี)

กด **"ส่ง"** ที่หน้าเว็บเพื่อบันทึกคะแนน